### Libraries

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

### Part I: Basics of LSTM in PyTorch

In [2]:
# create a simple lstm layer
lstm = nn.LSTM(input_size=5, hidden_size=10,num_layers=1,batch_first=True)
print(f'First LSTM layer created {lstm}')

First LSTM layer created LSTM(5, 10, batch_first=True)


In [3]:
# Let's see what happens when we pass data through it
batch_size = 2
sequence_length = 3
input_size = 5

# Create random input data
# Shape: (batch_size, sequence_length, input_size)
input_data = torch.rand(batch_size,sequence_length,input_size)
print(input_data)
print(f'shape of input is:{input_data.shape}')

tensor([[[0.3912, 0.6843, 0.7751, 0.3383, 0.2735],
         [0.8861, 0.0853, 0.8896, 0.8833, 0.8309],
         [0.9627, 0.3248, 0.5375, 0.4670, 0.7983]],

        [[0.2298, 0.7481, 0.8722, 0.9459, 0.0372],
         [0.6187, 0.6067, 0.8691, 0.7529, 0.7678],
         [0.7376, 0.8624, 0.5958, 0.5578, 0.5099]]])
shape of input is:torch.Size([2, 3, 5])


In [4]:
input_data[0][0]

tensor([0.3912, 0.6843, 0.7751, 0.3383, 0.2735])

In [5]:
# Forward pass through LSTM
output, (hidden, cell) = lstm(input_data)
print(f'output shape:{output.shape}')   # (batch, seq, hidden_size)
print(f'Final Hidden state shape:{hidden.shape}')   # (num_layers, batch, hidden_size)
print(f'Final cell state shape:{cell.shape}')

output shape:torch.Size([2, 3, 10])
Final Hidden state shape:torch.Size([1, 2, 10])
Final cell state shape:torch.Size([1, 2, 10])


### PART II: Simple Sequence Prediction

In [6]:
# create an sample dataset
def create_sine_dataset(seq_length, num_samples):
    """create a sine wave sequence for training"""
    X, y = [], []
    for i in range(num_samples):
        start = np.random.uniform(0,4*np.pi)
        x = np.linspace(start, 2*np.pi, seq_length+1)
        sine_vals = np.sin(x)

        # Use first seq_length values as input, last value as target
        X.append(sine_vals[:-1])
        y.append(sine_vals[-1])
    return torch.FloatTensor(X), torch.FloatTensor(y)


In [7]:
# create dataset
seq_length = 10
num_samples = 5
X_train, y_train = create_sine_dataset(seq_length, num_samples)
y_train.shape

C:\Users\Admin\AppData\Local\Temp\ipykernel_30184\1636833817.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  return torch.FloatTensor(X), torch.FloatTensor(y)


torch.Size([5])

In [8]:
print(X_train)
print(f"Training data shape: {X_train.shape}")
print(f"Training targets shape: {y_train.shape}")

tensor([[ 0.7870,  0.4804,  0.0962, -0.3035, -0.6542, -0.8992, -0.9988, -0.9371,
         -0.7239, -0.3938],
        [-0.9906, -0.9608, -0.9114, -0.8432, -0.7578, -0.6568, -0.5424, -0.4169,
         -0.2827, -0.1428],
        [-0.8383, -0.9353, -0.9892, -0.9977, -0.9604, -0.8790, -0.7572, -0.6006,
         -0.4164, -0.2131],
        [-0.2486, -0.5143, -0.7374, -0.8994, -0.9867, -0.9921, -0.9153, -0.7625,
         -0.5464, -0.2850],
        [-0.3605, -0.8193, -0.9998, -0.8406, -0.3959,  0.1834,  0.7003,  0.9794,
          0.9257,  0.5576]])
Training data shape: torch.Size([5, 10])
Training targets shape: torch.Size([5])


In [9]:
# Reshape input for LSTM (add feature dimension)
X_train = X_train.unsqueeze(-1)  # Add feature dimension
print(f"Reshaped input: {X_train.shape}")
print(X_train)

Reshaped input: torch.Size([5, 10, 1])
tensor([[[ 0.1422],
         [-0.1846],
         [-0.4918],
         [-0.7464],
         [-0.9211],
         [-0.9975],
         [-0.9671],
         [-0.8334],
         [-0.6106],
         [-0.3226]],

        [[-0.1869],
         [-0.1684],
         [-0.1498],
         [-0.1312],
         [-0.1126],
         [-0.0939],
         [-0.0751],
         [-0.0564],
         [-0.0376],
         [-0.0188]],

        [[-0.1893],
         [-0.4667],
         [-0.7038],
         [-0.8800],
         [-0.9801],
         [-0.9955],
         [-0.9248],
         [-0.7741],
         [-0.5566],
         [-0.2909]],

        [[-0.7416],
         [-0.9819],
         [-0.9379],
         [-0.6224],
         [-0.1267],
         [ 0.4057],
         [ 0.8206],
         [ 0.9980],
         [ 0.8864],
         [ 0.5182]],

        [[ 0.8876],
         [ 0.9992],
         [ 0.8475],
         [ 0.4725],
         [-0.0270],
         [-0.5193],
         [-0.8749],
         [-0.

### PART III: Building an LSTM Model

In [11]:
class simpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size,hidden_size,num_layers,batch_first=True)

        # fully connected layer to get the first output
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self,x):
        batch_size = x.size(0)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)

        # LSTM forward pass
        lstm_out, (hn, cn) = self.lstm(x, (h0, c0))
        
        # Use the last output for prediction
        last_output = lstm_out[:, -1, :]  # Get last time step
        # Pass through fully connected layer
        output = self.fc(last_output)
        
        return output

In [13]:
# Create model
model = simpleLSTM(input_size=1, hidden_size=50, num_layers=2, output_size=1)
print(f"Model created: {model}")
# Print model parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

Model created: simpleLSTM(
  (lstm): LSTM(1, 50, num_layers=2, batch_first=True)
  (fc): Linear(in_features=50, out_features=1, bias=True)
)
Total parameters: 31051


### PART IV: Training the Model

In [15]:
# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 100
losses = []

print("Training progress:")
for epoch in range(num_epochs):
    model.train()
    
    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs.squeeze(), y_train)
    
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Training progress:
Epoch [20/100], Loss: 0.0000
Epoch [40/100], Loss: 0.0000
Epoch [60/100], Loss: 0.0000
Epoch [80/100], Loss: 0.0000
Epoch [100/100], Loss: 0.0000
